# Alternate Experiments — Exercising the Decision Tree

The default `Checkout Flow Redesign` experiment lands on **HOLD** (revenue volatility breaches the guardrail). That's only one branch of `recommendation.py`'s decision tree. This notebook plugs two more `ExperimentConfig` + `TreatmentEffect` setups into the existing pipeline to exercise the other branches:

- **Experiment B — Forced Account Creation.** Engineered to be a bad idea: large negative effects on conversion and revenue. Expected verdict: **REJECT**.
- **Experiment C — Personalized Product Recommendations.** Engineered to be a clean win: moderate positive lifts, no guardrail spike. Expected verdict: **SHIP**.

Same pipeline, same modules, just different effect specs and seeds. The point is to show the platform behaving correctly across the full decision space and not js one run
We also use `power.py` here to ask whether each experiment is even powered to detect the effect it's looking for.

## Setup

In [ ]:
import sys
from pathlib import Path

SRC = Path.cwd().parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

from clean import build_clean_sessions
from assign_experiment import assign_users, ExperimentConfig
from simulate_treatment import simulate_treatment_effects, TreatmentEffect
from recommendation import make_recommendation
from power import analyze_experiment_power, required_sample_for_experiment


clean_df = build_clean_sessions()
print(f"Cleaned sessions ready: {len(clean_df):,}")

### Helper — run a full experiment cycle from a (config, effects) pair

In [ ]:
def run_experiment(clean_df, config, effects, label):
    """Assign → simulate → recommend. Returns dict with verdict, reasons, and df."""
    df = assign_users(clean_df, config=config)
    df = simulate_treatment_effects(df, effects=effects, seed=config.seed)
    rec = make_recommendation(df)
    return {
        "label": label,
        "verdict": rec["verdict"],
        "reasons": rec["reasons"],
        "summary": rec["summary"],
        "frequentist_primary": rec["frequentist"]["primary"],
        "guardrails": rec["guardrails"]["results"],
        "df": df,
    }

## Experiment B — Forced Account Creation Before Checkout

**Hypothesis:** Requiring users to create an account before checking out improves long-term value through better targeting.

**Reality (engineered):** Forcing signup is a well-known checkout-conversion killer. Users abandon when asked for friction they didn't expect.

**Ground truth:**
- −25% conversion
- −30% revenue per user
- −10% add-to-cart
- −10s session duration (users leave faster)
- +10% pageviews (some users hop around looking for alternatives)

Expected verdict: **REJECT** — primary metric significantly negative.

In [ ]:
config_b = ExperimentConfig(
    experiment_id="exp_002",
    experiment_name="Forced Account Creation Before Checkout",
    hypothesis="Requiring signup before checkout improves LTV via better targeting.",
    primary_metric="conversion_rate",
    guardrail_metrics=["bounce_rate", "session_depth", "revenue_volatility"],
    treatment_split=0.5,
    stratify_by=["device_type"],
    start_date="2020-12-01",
    end_date="2021-01-31",
    seed=99,
)

effects_b = [
    TreatmentEffect("converted", "multiplicative", 0.75),                
    TreatmentEffect("revenue", "multiplicative", 0.70),                  
    TreatmentEffect("add_to_cart_events", "multiplicative", 0.90),       #
    TreatmentEffect("session_duration_sec", "additive", -10.0),           
    TreatmentEffect("pageviews", "multiplicative", 1.10),                
]

exp_b = run_experiment(clean_df, config_b, effects_b, "B — Forced Account Creation")
print(exp_b["summary"])

## Experiment C — Personalized Product Recommendations

**Hypothesis:** ML-driven personalized recommendations on the homepage increase engagement and conversion without harming session quality.

**Reality (engineered):** A reasonable, conservative win. Moderate lifts across primary + engagement metrics, no metric coupling weirdness.

**Ground truth:**
- +8% conversion
- +6% revenue per user
- +15% add-to-cart
- +20% pageviews
- +30s session duration

Expected verdict: **SHIP** — primary significant + high posterior, guardrails clear.

In [ ]:
config_c = ExperimentConfig(
    experiment_id="exp_003",
    experiment_name="Personalized Product Recommendations",
    hypothesis="Personalized homepage recommendations increase conversion and engagement.",
    primary_metric="conversion_rate",
    guardrail_metrics=["bounce_rate", "session_depth", "revenue_volatility"],
    treatment_split=0.5,
    stratify_by=["device_type"],
    start_date="2020-12-01",
    end_date="2021-01-31",
    seed=123,
)

effects_c = [
    TreatmentEffect("converted", "multiplicative", 1.08),                
    TreatmentEffect("revenue", "multiplicative", 1.06),                  
    TreatmentEffect("add_to_cart_events", "multiplicative", 1.15),       
    TreatmentEffect("pageviews", "multiplicative", 1.20),                
    TreatmentEffect("session_duration_sec", "additive", 30.0),            
]

exp_c = run_experiment(clean_df, config_c, effects_c, "C — Personalized Recommendations")
print(exp_c["summary"])

## Comparison across all three experiments

Pull verdicts and primary-metric numbers side by side. The point is the **same code path** producing three different, correct verdicts.

In [ ]:
# Re-run the default experiment with the existing module defaults to round out the comparison.
from assign_experiment import DEFAULT_EXPERIMENT
from simulate_treatment import DEFAULT_EFFECTS
exp_a = run_experiment(clean_df, DEFAULT_EXPERIMENT, DEFAULT_EFFECTS, "A — Checkout Flow Redesign (default)")

verdict_table = pd.DataFrame([
    {"experiment": e["label"], "verdict": e["verdict"], "reasons": " | ".join(e["reasons"])}
    for e in [exp_a, exp_b, exp_c]
])
verdict_table

In [ ]:
# Primary-metric lift comparison
rows = []
for e in [exp_a, exp_b, exp_c]:
    for m in e["frequentist_primary"]:
        rows.append({
            "experiment": e["label"],
            "metric": m["metric_name"],
            "lift": m["relative_lift"],
            "p_value": m["p_value"],
            "significant": m["significant"],
        })
pd.DataFrame(rows).pivot(index="experiment", columns="metric", values="lift")

In [ ]:
# Guardrail comparison
rows = []
for e in [exp_a, exp_b, exp_c]:
    for g in e["guardrails"]:
        rows.append({
            "experiment": e["label"],
            "guardrail": g["name"],
            "rel_diff": g["relative_diff"],
            "status": g["status"],
        })
pd.DataFrame(rows).pivot(index="experiment", columns="guardrail", values="status")

## Power analysis — were these experiments adequately powered?

Before running an experiment, you'd ask: *given the n I'll get, what's the smallest effect I can detect at 80% power?* If the MDE is bigger than what you're hoping to find, you're underpowered and any null result is uninformative.

We run `analyze_experiment_power` on each experiment's df. Compare the **MDE relative** column to the **ground-truth lift** you injected — anywhere the MDE exceeds the truth, you're at the edge of detectability.

In [ ]:
rows = []
for e in [exp_a, exp_b, exp_c]:
    for r in analyze_experiment_power(e["df"]):
        rows.append({
            "experiment": e["label"],
            "metric": r.metric_name,
            "baseline": round(r.baseline, 5),
            "n_per_arm": r.n_per_arm_observed,
            "MDE_absolute": round(r.mde_absolute, 6),
            "MDE_relative": f"{r.mde_relative:.2%}",
        })
pd.DataFrame(rows)

## Pre-experiment sizing — how many users would we need to detect a +3% conversion lift?

Before launching, sizing the experiment correctly is what avoids wasted runs. Pass target relative lifts per metric and `required_sample_for_experiment` returns users-per-arm needed at 80% power.

In [ ]:
required_sample_for_experiment(
    exp_a["df"],
    target_relative_lifts={
        "conversion_rate": 0.03,    # detect +3%
        "revenue_per_user": 0.05,   # detect +5%
    },
)

## Takeaway

The same modules — `clean`, `assign_users`, `simulate_treatment_effects`, `make_recommendation` — produce three correct, distinct verdicts for three different experiments:

- **A (default) → HOLD** — topline good, volatility guardrail fails
- **B (forced account creation) → REJECT** — primary metric significantly negative
- **C (personalized recommendations) → SHIP** — primary positive, high posterior, guardrails clear

The platform's value isn't recovering known ground truth on a single experiment. It's having a *deterministic, auditable* path from raw data to a defensible decision across the full space of outcomes. Whether the experiment is a hit, a flop, or risky, the same pipeline gives a verdict you can defend in a review meeting.

Power analysis tells you whether the experiment can answer the question at all — a critical pre-flight step that's separate from the post-hoc inference layer.